# Preprocessing: Annotate Event
Tests for event selection and annotation functions

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'

## Create test data
Create a simple test DataFrame with multiple surgery events

In [ ]:
# Create test data with multiple surgery events
test_data = pd.DataFrame({
    'patient_id': [1, 2, 3, 4],
    'diagnosis_date': pd.to_datetime([
        '2020-01-10', '2020-02-10', '2020-03-10', '2020-01-15'
    ], utc=True),
    'surgery_1_date': pd.to_datetime([
        '2020-01-15', '2020-02-20', None, '2020-01-20'
    ], utc=True),
    'surgery_1_intent': ['curative', 'palliative', None, 'curative'],
    'surgery_1_margins_after_surgery': ['negative', 'positive', None, 'negative'],
    'surgery_2_date': pd.to_datetime([
        '2020-03-15', None, '2020-04-10', '2020-03-20'
    ], utc=True),
    'surgery_2_intent': ['curative', None, 'palliative', 'curative'],
    'surgery_2_margins_after_surgery': ['negative', None, 'positive', 'negative'],
    'surgery_3_date': pd.to_datetime([
        None, None, None, '2020-05-10'
    ], utc=True),
    'surgery_3_intent': [None, None, None, 'curative'],
    'surgery_3_margins_after_surgery': [None, None, None, 'negative'],
})

print("Test DataFrame:")
print(test_data[['patient_id', 'diagnosis_date', 'surgery_1_date', 'surgery_2_date', 'surgery_3_date']])

## Setup MockNetwork

In [ ]:
from vantage6.algorithm.mock.network import MockNetwork

network = MockNetwork(
    "v6_preprocessing",
    datasets=[
        {
            "cohort_1": {"database": test_data.iloc[:2], "db_type": "omop"},
        },
        {
            "cohort_1": {"database": test_data.iloc[2:], "db_type": "omop"},
        }
    ],
    collaboration_id=1,
)

client = network.user_client

## Test 1: annotate_event_by_index with first surgery

In [ ]:
result = client.dataframe.preprocess(
    id_=1,
    method="annotate_event_by_index",
    image="v6-preprocessing",
    arguments={
        "type_": "surgery",
        "index": 1,
        "name": "primary"
    }
)

# Show the annotated columns
df1 = client.network.get_node(1).dataframes["cohort_1"]
print("\nNode 1 - Primary surgery (first):")
print(df1[['patient_id', 'primary_surgery_date', 'primary_surgery_intent']])

## Test 2: annotate_event_by_index with last surgery

In [ ]:
result = client.dataframe.preprocess(
    id_=2,
    method="annotate_event_by_index",
    image="v6-preprocessing",
    arguments={
        "type_": "surgery",
        "index": "last",
        "name": "final"
    }
)

df2 = client.network.get_node(2).dataframes["cohort_1"]
print("\nNode 2 - Final surgery (last):")
print(df2[['patient_id', 'final_surgery_date', 'final_surgery_intent']])

## Test 3: annotate_event_by_date_range with date strings

In [ ]:
result = client.dataframe.preprocess(
    id_=3,
    method="annotate_event_by_date_range",
    image="v6-preprocessing",
    arguments={
        "type_": "surgery",
        "name": "target_range",
        "start_date_string": "2020-02-01",
        "end_date_string": "2020-04-30"
    }
)

print("\nEvent by date range (2020-02-01 to 2020-04-30):")
print("This test would need both nodes to show results with the mock network setup.")
print("The function searches for the first surgery within the specified date range.")

## Test 4: annotate_event_within_window with reference column

In [ ]:
result = client.dataframe.preprocess(
    id_=4,
    method="annotate_event_within_window",
    image="v6-preprocessing",
    arguments={
        "type_": "surgery",
        "name": "perioperative",
        "reference_date_column": "diagnosis_date",
        "days_before": 30,
        "days_after": 90
    }
)

print("\nEvent within window (diagnosis_date ± 30/90 days):")
print("This function finds the first event within a time window relative to a reference date.")